# Notebook 6 v2 — Calibration, confidence gate and correction decision (shift-aware)

**What this notebook does:** turns classifier probabilities into **decisions** (`no_action`, `preserve`, `correct`, `defer_flag`), and measures how safe those decisions are, including when the event direction or the drift type changes.

## Why a new version (problems found in the first NB6 run)
| First NB6 | Problem | NB6 v2 |
|---|---|---|
| Temperature scaling bounded at 0.5 to 5 | Every temperature landed on the upper bound, and the "calibrated" log loss (1.5 to 1.7) was worse than uniform guessing (1.386) | Compare **raw, temperature, isotonic and Platt**, chosen by **cross-fitting over validation episodes**; must beat both raw and uniform, otherwise it is flagged |
| Calibrator evaluated on the same validation data it was fitted on | Over-optimistic | Calibrated validation probabilities for threshold selection are **cross-fitted by episode** |
| 70% coverage floor | Infeasible, so the gate fell back to a threshold that accepted about 10% of windows | Explicit rule: **maximum coverage subject to a target selective error**, plus full risk-versus-coverage tables |
| Correction gate | "0% false correction" because it almost never corrected | **Correction rate and false-correction rate reported together**, by true class, with confidence intervals |
| All feature sets treated alike | Ignored the NB5 v2 finding that level-based sets collapse on cooling events | **G** (no level features) is primary, **F** is the reference, and **I** (G + \|peer_level_rel\|) is an **exploratory, post-hoc** direction-independent set |

## Protocol
- Models are fitted on **train** episodes only; calibrators and thresholds are fitted on **validation** episodes only; the test episodes are scored once at the end.
- Two regimes: **W** (train and validate on warming + ramp variants only; the real cross-condition test) and **ALL** (all nine variants; reference).
- Four test conditions, as in NB5 v2: `baseline`, `cool_ramp`, `warm_step`, `cool_step`.
- RF settings are **fixed** (500 trees, `min_samples_leaf=5`, balanced class weights) to avoid tuning and to avoid the zero-probability leaves that broke the first calibration.

## Decision policy
| Condition on the (calibrated) probabilities | Decision |
|---|---|
| predicted DRIFT and P(DRIFT) ≥ `t_corr` | `correct` |
| confidence ≥ `tau` and predicted NORMAL | `no_action` |
| confidence ≥ `tau` and predicted EVENT | `preserve` |
| everything else, including predicted EVENT_DRIFT | `defer_flag` |

EVENT_DRIFT is not corrected here. A bounded correction for it is a design question for NB7. This notebook evaluates the **decision gate**, not a correction magnitude.

**Limits:** 4 test episodes; windows overlap 83%, so intervals are descriptive; synthetic injections; the test-period voltage shift is not handled; thresholds chosen on validation may not transfer under shift, and the tables show exactly how much they do not.

In [2]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
try:
    import sklearn
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.isotonic import IsotonicRegression
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import f1_score
    from scipy.optimize import minimize_scalar
except ImportError:
    raise ImportError("scikit-learn / scipy missing. In your Jupyter environment run: conda install scikit-learn scipy  (then restart the kernel)")

_here = Path.cwd().resolve()
ROOT = next((p for p in [_here, *_here.parents] if (p / "data" / "processed").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find data/processed above " + str(_here) + ". Start Jupyter inside the project folder.")
PROC = ROOT / "data" / "processed"
OUT = ROOT / "outputs" / "notebook06_v2"      # the first NB6 outputs (outputs/notebook06) are not touched
OUT.mkdir(parents=True, exist_ok=True)

def check(cond, msg):
    if not bool(cond):
        raise ValueError("CHECK FAILED: " + msg)
    print("PASS ", msg)

def warn(cond, msg):
    print("OK   " if cond else "WARN ", msg)
print("ROOT:", ROOT, "| sklearn", sklearn.__version__, "| pandas", pd.__version__)

ROOT: C:\AIoT_Research | sklearn 1.9.1 | pandas 2.2.2


In [3]:
QUICK_TEST = False          # True = small forests, just to check the notebook runs; use False for the real results
LABELS = ["NORMAL", "DRIFT", "EVENT", "EVENT_DRIFT"]
N_TREES = 60 if QUICK_TEST else 500
RF_PARAMS = {"min_samples_leaf": 5, "max_features": "sqrt"}
SEED = 0
CAL_METHODS = ["raw", "temperature", "isotonic", "platt"]
EPS = 1e-4                  # probability floor before any log (stops zero-probability leaves from giving infinite loss)
THR_GRID = np.round(np.arange(0.30, 0.995, 0.01), 2)
TARGET_RISK = 0.20          # abstention gate: highest coverage whose validation selective error is at most this
MIN_ACCEPTED = 30           # a threshold must accept at least this many validation windows to count
MAX_FALSE_CORRECTION = 0.05 # correction gate: false-correction rate on validation windows that are not DRIFT
VARIANTS = ["NORMAL", "DRIFT_RAMP", "DRIFT_STEP", "EVENT_WARM", "EVENT_COOL", "ED_WARM_RAMP", "ED_WARM_STEP", "ED_COOL_RAMP", "ED_COOL_STEP"]
W_TRAIN = ["NORMAL", "DRIFT_RAMP", "EVENT_WARM", "ED_WARM_RAMP"]
CONDITIONS = {
    "baseline":  ["NORMAL", "DRIFT_RAMP", "EVENT_WARM", "ED_WARM_RAMP"],
    "cool_ramp": ["NORMAL", "DRIFT_RAMP", "EVENT_COOL", "ED_COOL_RAMP"],
    "warm_step": ["NORMAL", "DRIFT_STEP", "EVENT_WARM", "ED_WARM_STEP"],
    "cool_step": ["NORMAL", "DRIFT_STEP", "EVENT_COOL", "ED_COOL_STEP"],
}
if QUICK_TEST:
    print("QUICK_TEST is ON: results are for checking the code only, NOT for reporting")

## 1. Load the NB4 v2 table and define the feature sets

In [5]:
T = pd.read_parquet(PROC / "feature_dataset_AF_v2.parquet")
with open(PROC / "feature_config_AF_v2.json") as f:
    FC = json.load(f)
A_FEATURES, F_FEATURES = FC["A_features"], FC["F_features"]
G_FEATURES = [f for f in F_FEATURES if f != "peer_level_rel"]
T["abs_peer_level_rel"] = T["peer_level_rel"].abs()              # exploratory direction-independent feature (set I)
I_FEATURES = G_FEATURES + ["abs_peer_level_rel"]
SETS = {"G": G_FEATURES, "F": F_FEATURES, "I": I_FEATURES}      # G = primary, F = reference, I = exploratory / post-hoc
ALL_FEATURES = FC["all_features"] + ["abs_peer_level_rel"]
PURE = T[T["window_pure"]]

def rows(split, variants=None):
    m = PURE["split"] == split
    if variants is not None:
        m &= PURE["variant"].isin(variants)
    return PURE[m]

DATA = {"W":   {"train": rows("train", W_TRAIN), "val": rows("validation", W_TRAIN)},
        "ALL": {"train": rows("train"),          "val": rows("validation")}}
TE = rows("test")
print({k: len(v) for k, v in SETS.items()}, "features per set (G primary, F reference, I exploratory)")
print("rows  W: train", len(DATA["W"]["train"]), "val", len(DATA["W"]["val"]), "| ALL: train", len(DATA["ALL"]["train"]), "val", len(DATA["ALL"]["val"]), "| test (all variants):", len(TE))

{'G': 14, 'F': 15, 'I': 15} features per set (G primary, F reference, I exploratory)
rows  W: train 2770 val 1108 | ALL: train 6170 val 2468 | test (all variants): 2468


## 2. Data checks

In [7]:
check(len(G_FEATURES) == 14 and len(F_FEATURES) == 15 and len(I_FEATURES) == 15, "feature set sizes: G 14, F 15, I 15")
check(set(I_FEATURES) == set(G_FEATURES) | {"abs_peer_level_rel"}, "I equals G plus abs_peer_level_rel only")
LEAK = {"variant", "event_direction", "drift_type", "drift_event_relation", "interval_phase", "label", "scenario", "episode_id", "split", "label_purity",
        "long_label_purity", "window_pure", "long_window_pure", "window_group", "minutes_since_onset", "gt_event_component", "gt_drift_component",
        "offset_used", "baseline_used", "end_index"}
check(all(not (LEAK & set(v)) for v in SETS.values()), "no label / variant / ground-truth column inside any feature set")
check(set(T["variant"]) == set(VARIANTS) and set(T["label"]) == set(LABELS), "9 variants and 4 labels present")
eps = T.groupby("split")["episode_id"].unique().apply(set); sp = list(eps.index)
check(all(eps[a].isdisjoint(eps[b]) for i, a in enumerate(sp) for b in sp[i + 1:]), "no episode appears in two splits")
check(DATA["W"]["val"]["episode_id"].nunique() >= 3, "validation has at least 3 episodes (needed for cross-fitted calibration)")
for k, vs in CONDITIONS.items():
    d = TE[TE["variant"].isin(vs)]
    check(set(d["label"]) == set(LABELS), f"test condition '{k}': all four classes present ({len(d)} windows)")
print("validation episodes:", sorted(DATA["W"]["val"]["episode_id"].unique()), "| test episodes:", sorted(TE["episode_id"].unique()))

PASS  feature set sizes: G 14, F 15, I 15
PASS  I equals G plus abs_peer_level_rel only
PASS  no label / variant / ground-truth column inside any feature set
PASS  9 variants and 4 labels present
PASS  no episode appears in two splits
PASS  validation has at least 3 episodes (needed for cross-fitted calibration)
PASS  test condition 'baseline': all four classes present (1108 windows)
PASS  test condition 'cool_ramp': all four classes present (1108 windows)
PASS  test condition 'warm_step': all four classes present (1108 windows)
PASS  test condition 'cool_step': all four classes present (1108 windows)
validation episodes: ['E11', 'E12', 'E13', 'E14'] | test episodes: ['E15', 'E16', 'E17', 'E18']


## 3. Helpers: calibrators, metrics, gates, decisions
Calibrators are fitted on probability matrices (rows = windows, columns in `LABELS` order). Every probability is floored at 1e-4 before any logarithm, so a zero-probability leaf cannot create an infinite loss.

In [9]:
def norm(P):
    P = np.clip(np.asarray(P, float), EPS, 1.0)
    return P / P.sum(axis=1, keepdims=True)

def logit(p):
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p))

def onehot(y):
    return (np.asarray(y)[:, None] == np.array(LABELS)[None, :]).astype(float)

def log_loss_(y, P):
    P = norm(P); idx = pd.Categorical(y, categories=LABELS).codes
    return float(-np.mean(np.log(P[np.arange(len(P)), idx])))

def brier_(y, P):
    return float(np.mean(np.sum((norm(P) - onehot(y)) ** 2, axis=1)))

def ece_(y, P, bins=10):
    P = norm(P); conf = P.max(axis=1); ok = (np.array(LABELS)[P.argmax(axis=1)] == np.asarray(y)).astype(float)
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(ok[m].mean() - conf[m].mean())
    return float(e)

UNIFORM_LOGLOSS, UNIFORM_BRIER = float(np.log(4)), 0.75       # what predicting 25% for every class scores

def softmax_T(P, T_):
    z = np.log(norm(P)) / T_; z -= z.max(axis=1, keepdims=True); Q = np.exp(z)
    return Q / Q.sum(axis=1, keepdims=True)

def fit_calibrator(method, P, y):
    P = norm(P)
    if method == "raw":
        return None
    if method == "temperature":
        return float(minimize_scalar(lambda t: log_loss_(y, softmax_T(P, t)), bounds=(0.2, 10.0), method="bounded").x)
    if method == "isotonic":
        return [IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip").fit(P[:, k], onehot(y)[:, k]) for k in range(4)]
    if method == "platt":
        return [LogisticRegression(C=100.0, max_iter=2000).fit(logit(P[:, k]).reshape(-1, 1), onehot(y)[:, k].astype(int)) for k in range(4)]
    raise ValueError(method)

def apply_calibrator(method, params, P):
    P = norm(P)
    if method == "raw":
        return P
    if method == "temperature":
        return softmax_T(P, params)
    if method == "isotonic":
        return norm(np.column_stack([params[k].predict(P[:, k]) for k in range(4)]))
    if method == "platt":
        return norm(np.column_stack([params[k].predict_proba(logit(P[:, k]).reshape(-1, 1))[:, 1] for k in range(4)]))
    raise ValueError(method)

def crossfit(method, P, y, groups):
    # calibrate each validation episode with a calibrator fitted on the OTHER validation episodes
    out = np.zeros_like(norm(P))
    for g in np.unique(groups):
        te = groups == g
        out[te] = apply_calibrator(method, fit_calibrator(method, P[~te], y[~te]), P[te])
    return out

def pred_of(P):
    return np.array(LABELS)[np.asarray(P).argmax(axis=1)]

def select_tau(P, y, target_risk):
    # highest coverage (lowest threshold) whose selective error on validation is <= target_risk and that accepts enough windows
    conf, pred = P.max(axis=1), pred_of(P)
    for t in THR_GRID:
        acc = conf >= t
        if acc.sum() >= MIN_ACCEPTED and (pred[acc] != y[acc]).mean() <= target_risk:
            return float(t), float(acc.mean()), float((pred[acc] != y[acc]).mean())
    return 1.01, 0.0, float("nan")                    # no feasible threshold: abstain on everything

def select_tcorr(P, y, alpha):
    # correction threshold: highest correction rate on true DRIFT windows subject to false-correction rate <= alpha on all other windows
    pred, pd_ = pred_of(P), P[:, LABELS.index("DRIFT")]
    non = y != "DRIFT"; best = None
    for t in THR_GRID:
        att = (pred == "DRIFT") & (pd_ >= t)
        fcr, rate = float(att[non].mean()), float(att[~non].mean())
        if fcr <= alpha and (best is None or rate > best[2] + 1e-12):
            best = (float(t), fcr, rate)
    return best if best is not None else (1.01, 0.0, 0.0)

def decide(P, tau, t_corr):
    pred, conf, pd_ = pred_of(P), P.max(axis=1), P[:, LABELS.index("DRIFT")]
    dec = np.full(len(P), "defer_flag", dtype=object)
    acc = conf >= tau
    dec[acc & (pred == "NORMAL")] = "no_action"
    dec[acc & (pred == "EVENT")] = "preserve"
    dec[(pred == "DRIFT") & (pd_ >= t_corr)] = "correct"
    return dec

def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n; den = 1 + z * z / n; c = (p + z * z / (2 * n)) / den
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (c - h, c + h)
print("helpers defined | uniform reference: log loss %.3f, Brier %.2f" % (UNIFORM_LOGLOSS, UNIFORM_BRIER))

helpers defined | uniform reference: log loss 1.386, Brier 0.75


## 4. Fit models, cross-fit calibration on validation, select thresholds
For each regime and feature set: fit the forest on train; produce raw validation and test probabilities; evaluate every calibration method **cross-fitted over validation episodes**; choose the method with the lowest cross-fitted log loss; fit the final calibrator on all validation episodes; select `tau` and `t_corr` on the **cross-fitted** calibrated validation probabilities (so they are not tuned on probabilities the calibrator has already seen).

In [43]:

def pipeline(Tfull, regime, feats):
    pure = Tfull[Tfull["window_pure"]]

    def rw(split, variants=None):
        m = pure["split"] == split
        if variants is not None:
            m &= pure["variant"].isin(variants)
        return pure[m]

    tr = rw("train", W_TRAIN if regime == "W" else None)
    va = rw("validation", W_TRAIN if regime == "W" else None)
    te = rw("test")

    med = tr[feats].median()
    assert med.notna().all()

    rf = RandomForestClassifier(
        n_estimators=N_TREES,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,
        **RF_PARAMS
    )
    rf.fit(
        tr[feats].fillna(med).to_numpy(float),
        tr["label"].to_numpy()
    )
    assert set(rf.classes_) == set(LABELS)

    def prob(d):
        return pd.DataFrame(
            rf.predict_proba(d[feats].fillna(med).to_numpy(float)),
            columns=rf.classes_,
            index=d.index
        )[LABELS].to_numpy()

    Pv, Pt = prob(va), prob(te)

    yv = va["label"].to_numpy()
    gv = va["episode_id"].to_numpy()

    cf = {
        m: crossfit(m, Pv, yv, gv)
        for m in CAL_METHODS
    }

    cfm = {
        m: {
            "log_loss": log_loss_(yv, cf[m]),
            "brier": brier_(yv, cf[m]),
            "ece": ece_(yv, cf[m])
        }
        for m in CAL_METHODS
    }

    best = min(CAL_METHODS, key=lambda m: cfm[m]["log_loss"])
    params = fit_calibrator(best, Pv, yv)

    Pv_cf = cf[best]
    Pt_cal = apply_calibrator(best, params, Pt)

    tau, cov_v, risk_v = select_tau(Pv_cf, yv, TARGET_RISK)
    t_corr, fcr_v, rate_v = select_tcorr(
        Pv_cf, yv, MAX_FALSE_CORRECTION
    )

    return {
        "val": va,
        "test": te,
        "P_val_raw": Pv,
        "P_val_cf": Pv_cf,
        "P_test_raw": Pt,
        "P_test_cal": Pt_cal,
        "method": best,
        "params": params,
        "crossfit": cfm,
        "tau": tau,
        "val_cov": cov_v,
        "val_risk": risk_v,
        "t_corr": t_corr,
        "val_fcr": fcr_v,
        "val_corr_rate": rate_v,

        # Retain fitted inference components for NB8 profiling.
        "rf": rf,
        "med": med.copy(),
        "features": list(feats),
    }


PIPE = {}

for regime in ["W", "ALL"]:
    for s, feats in SETS.items():
        PIPE[(regime, s)] = pipeline(T, regime, feats)
        pl = PIPE[(regime, s)]

        print(
            f"{regime:>3}/{s}: "
            f"calibrator = {pl['method']:<11} | "
            f"tau = {pl['tau']:.2f} "
            f"(validation coverage {pl['val_cov']:.3f}, "
            f"risk {pl['val_risk']:.3f}) | "
            f"t_corr = {pl['t_corr']:.2f} "
            f"(validation false-correction {pl['val_fcr']:.3f}, "
            f"drift corrected {pl['val_corr_rate']:.3f})"
        )


# Export fitted models and inference metadata for NB8.
# These are additional artifacts; existing prediction and calibration
# outputs remain unchanged.
import joblib

MODEL_DIR = ROOT / "outputs" / "notebook06_v2" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

for (regime, s), pl in PIPE.items():
    artifact = {
        "regime": regime,
        "feature_set": s,
        "model": pl["rf"],
        "imputation_medians": pl["med"].to_dict(),
        "features": pl["features"],
        "classes": list(pl["rf"].classes_),
        "calibration_method": pl["method"],
        "calibration_params": pl["params"],
        "tau": float(pl["tau"]),
        "t_corr": float(pl["t_corr"]),
        "random_state": SEED,
    }

    out_path = MODEL_DIR / f"nb6_{regime}_{s}.joblib"
    joblib.dump(artifact, out_path)
    print(f"Saved {regime}/{s}: {out_path}")

print(f"\nInference artifacts saved to: {MODEL_DIR}")
print(f"Total artifacts saved: {len(PIPE)}")

  W/G: calibrator = raw         | tau = 0.65 (validation coverage 0.332, risk 0.196) | t_corr = 0.52 (validation false-correction 0.043, drift corrected 0.451)
  W/F: calibrator = raw         | tau = 0.62 (validation coverage 0.412, risk 0.200) | t_corr = 0.49 (validation false-correction 0.049, drift corrected 0.399)
  W/I: calibrator = raw         | tau = 0.67 (validation coverage 0.292, risk 0.198) | t_corr = 0.40 (validation false-correction 0.048, drift corrected 0.459)
ALL/G: calibrator = platt       | tau = 0.70 (validation coverage 0.207, risk 0.193) | t_corr = 0.30 (validation false-correction 0.033, drift corrected 0.285)
ALL/F: calibrator = platt       | tau = 0.72 (validation coverage 0.254, risk 0.199) | t_corr = 0.45 (validation false-correction 0.050, drift corrected 0.326)
ALL/I: calibrator = platt       | tau = 0.72 (validation coverage 0.235, risk 0.196) | t_corr = 0.47 (validation false-correction 0.048, drift corrected 0.257)
Saved W/G: C:\AIoT_Research\outputs\note

In [12]:
# How much coverage is available at each error target, on validation (cross-fitted, calibrated)? Use this to judge whether TARGET_RISK is realistic.
rows_ = []
for (regime, s), pl in PIPE.items():
    yv = pl["val"]["label"].to_numpy(); err_full = float((pred_of(pl["P_val_cf"]) != yv).mean())
    for r in [0.10, 0.20, 0.30, 0.40, 0.50]:
        t, cov, rk = select_tau(pl["P_val_cf"], yv, r)
        rows_.append({"regime": regime, "set": s, "target_risk": r, "tau": t, "coverage": cov, "error_at_full_coverage": err_full})
FRONT = pd.DataFrame(rows_)
print("Validation coverage reachable at each selective-error target (0.000 = no threshold reaches that target):")
print(FRONT.pivot_table(index=["regime", "set"], columns="target_risk", values="coverage").round(3).to_string())
print("\nError if nothing is rejected (full coverage):", FRONT.groupby(["regime", "set"])["error_at_full_coverage"].first().round(3).to_dict())
for (regime, s), pl in PIPE.items():
    warn(pl["tau"] <= 1.0, f"{regime}/{s}: abstention gate accepts some windows at TARGET_RISK={TARGET_RISK} (otherwise it abstains on everything: raise TARGET_RISK using the table above, or accept that confidence is not informative)")

Validation coverage reachable at each selective-error target (0.000 = no threshold reaches that target):
target_risk    0.1    0.2    0.3    0.4    0.5
regime set                                    
ALL    F     0.045  0.254  0.550  0.959  0.994
       G     0.058  0.207  0.481  0.976  0.996
       I     0.107  0.235  0.446  0.790  0.987
W      F     0.192  0.412  0.616  0.965  0.979
       G     0.206  0.332  0.510  0.748  0.977
       I     0.183  0.292  0.467  0.771  0.981

Error if nothing is rejected (full coverage): {('ALL', 'F'): 0.408, ('ALL', 'G'): 0.402, ('ALL', 'I'): 0.453, ('W', 'F'): 0.408, ('W', 'G'): 0.471, ('W', 'I'): 0.462}
OK    W/G: abstention gate accepts some windows at TARGET_RISK=0.2 (otherwise it abstains on everything: raise TARGET_RISK using the table above, or accept that confidence is not informative)
OK    W/F: abstention gate accepts some windows at TARGET_RISK=0.2 (otherwise it abstains on everything: raise TARGET_RISK using the table above, or accept tha

## 5. Checks: train-only fitting, no test leakage, reproducibility

In [14]:
# 5.1 corrupt every TEST feature value, rerun the whole pipeline: validation probabilities, calibrator choice and thresholds must not change
T_bad = T.copy(); T_bad.loc[T_bad["split"] == "test", ALL_FEATURES] = 999.0
for regime, s in [("W", "G"), ("W", "F")]:
    a, b = PIPE[(regime, s)], pipeline(T_bad, regime, SETS[s])
    check(a["method"] == b["method"] and a["tau"] == b["tau"] and a["t_corr"] == b["t_corr"], f"{regime}/{s}: calibrator choice and thresholds unchanged when all test features are garbage")
    check(np.allclose(a["P_val_cf"], b["P_val_cf"]) and np.allclose(a["P_val_raw"], b["P_val_raw"]), f"{regime}/{s}: validation probabilities unchanged when all test features are garbage")
# 5.2 corrupt VALIDATION features: the forest (trained on train only) must be unchanged, so test raw probabilities stay identical
T_badv = T.copy(); T_badv.loc[T_badv["split"] == "validation", ALL_FEATURES] = 999.0
b = pipeline(T_badv, "W", SETS["G"])
check(np.allclose(PIPE[("W", "G")]["P_test_raw"], b["P_test_raw"]), "W/G: raw test probabilities unchanged when validation features are garbage (the forest uses train data only)")
# 5.3 reproducibility
again = pipeline(T, "W", SETS["G"])
check(np.allclose(again["P_test_cal"], PIPE[("W", "G")]["P_test_cal"]) and again["tau"] == PIPE[("W", "G")]["tau"], "re-running W/G reproduces calibrated test probabilities and thresholds exactly")
# 5.4 probability sanity
for key, pl in PIPE.items():
    for nm in ["P_val_cf", "P_test_raw", "P_test_cal"]:
        P = pl[nm]
        assert np.isfinite(P).all() and np.allclose(P.sum(axis=1), 1.0, atol=1e-6) and (P >= 0).all(), f"{key} {nm}"
check(True, "all probability matrices are finite, non-negative and sum to 1")
check(all(pl["val"]["split"].eq("validation").all() and pl["test"]["split"].eq("test").all() for pl in PIPE.values()), "validation rows are validation episodes, test rows are test episodes")

PASS  W/G: calibrator choice and thresholds unchanged when all test features are garbage
PASS  W/G: validation probabilities unchanged when all test features are garbage
PASS  W/F: calibrator choice and thresholds unchanged when all test features are garbage
PASS  W/F: validation probabilities unchanged when all test features are garbage
PASS  W/G: raw test probabilities unchanged when validation features are garbage (the forest uses train data only)
PASS  re-running W/G reproduces calibrated test probabilities and thresholds exactly
PASS  all probability matrices are finite, non-negative and sum to 1
PASS  validation rows are validation episodes, test rows are test episodes


## 6. Calibration: does it help, and does it survive a shift?
First, the cross-fitted comparison on validation (each episode calibrated by a calibrator fitted on the other validation episodes). **A method only counts if it beats raw and uniform guessing.**

In [16]:
rows_ = []
for (regime, s), pl in PIPE.items():
    for m in CAL_METHODS:
        rows_.append({"regime": regime, "set": s, "method": m, **pl["crossfit"][m], "chosen": m == pl["method"]})
CF = pd.DataFrame(rows_)
print(f"Uniform guessing: log loss {UNIFORM_LOGLOSS:.3f}, Brier {UNIFORM_BRIER:.2f}.  Cross-fitted validation scores (lower is better):")
print(CF.round(3).to_string(index=False))
for (regime, s), pl in PIPE.items():
    best = pl["crossfit"][pl["method"]]; raw = pl["crossfit"]["raw"]
    warn(best["log_loss"] < UNIFORM_LOGLOSS and best["log_loss"] < raw["log_loss"],
         f"{regime}/{s}: chosen calibrator '{pl['method']}' beats raw ({best['log_loss']:.3f} vs {raw['log_loss']:.3f}) and uniform ({UNIFORM_LOGLOSS:.3f}) in log loss")

Uniform guessing: log loss 1.386, Brier 0.75.  Cross-fitted validation scores (lower is better):
regime set      method  log_loss  brier   ece  chosen
     W   G         raw     1.044  0.566 0.067    True
     W   G temperature     1.071  0.579 0.042   False
     W   G    isotonic     1.070  0.568 0.047   False
     W   G       platt     1.090  0.586 0.050   False
     W   F         raw     0.950  0.524 0.029    True
     W   F temperature     0.950  0.525 0.032   False
     W   F    isotonic     1.056  0.552 0.045   False
     W   F       platt     0.968  0.531 0.052   False
     W   I         raw     1.054  0.572 0.053    True
     W   I temperature     1.058  0.571 0.044   False
     W   I    isotonic     1.156  0.582 0.022   False
     W   I       platt     1.068  0.570 0.046   False
   ALL   G         raw     0.975  0.549 0.033   False
   ALL   G temperature     0.979  0.551 0.032   False
   ALL   G    isotonic     0.992  0.547 0.034   False
   ALL   G       platt     0.957  0.535

In [17]:
# Test calibration quality per condition: raw vs chosen calibrator. Under a shift, 'confidence' stops meaning what it meant on validation.
rows_ = []
for (regime, s), pl in PIPE.items():
    te = pl["test"]
    for cond, vs in CONDITIONS.items():
        m = te["variant"].isin(vs).to_numpy(); y = te.loc[m, "label"].to_numpy()
        for name, P in [("raw", norm(pl["P_test_raw"][m])), ("calibrated", pl["P_test_cal"][m])]:
            rows_.append({"regime": regime, "set": s, "condition": cond, "probs": name, "log_loss": log_loss_(y, P), "brier": brier_(y, P), "ece": ece_(y, P),
                          "mean_confidence": float(P.max(axis=1).mean()), "accuracy": float((pred_of(P) == y).mean())})
CALT = pd.DataFrame(rows_)
print("Test probability quality (regime W is the cross-condition test). 'mean_confidence' above 'accuracy' means over-confident:")
print(CALT[(CALT["regime"] == "W") & (CALT["probs"] == "calibrated")].drop(columns=["regime", "probs"]).round(3).to_string(index=False))
print("\\nRegime ALL (reference):")
print(CALT[(CALT["regime"] == "ALL") & (CALT["probs"] == "calibrated")].drop(columns=["regime", "probs"]).round(3).to_string(index=False))

Test probability quality (regime W is the cross-condition test). 'mean_confidence' above 'accuracy' means over-confident:
set condition  log_loss  brier   ece  mean_confidence  accuracy
  G  baseline     1.052  0.583 0.043            0.563     0.531
  G cool_ramp     1.040  0.599 0.053            0.569     0.524
  G warm_step     0.886  0.510 0.030            0.578     0.589
  G cool_step     0.916  0.534 0.061            0.593     0.565
  F  baseline     0.870  0.481 0.063            0.596     0.653
  F cool_ramp     1.145  0.643 0.143            0.581     0.439
  F warm_step     0.724  0.392 0.126            0.608     0.725
  F cool_step     1.020  0.588 0.085            0.592     0.507
  I  baseline     0.971  0.543 0.048            0.591     0.589
  I cool_ramp     0.865  0.482 0.053            0.596     0.639
  I warm_step     0.817  0.450 0.070            0.599     0.662
  I cool_step     0.761  0.417 0.117            0.604     0.718
\nRegime ALL (reference):
set condition  log_l

## 7. Abstention gate: coverage versus error
`tau` is the lowest confidence threshold whose validation selective error is at most the target (`TARGET_RISK`). The first table shows what that threshold does on each test condition. The second ranks test windows by confidence and shows the error at fixed coverage, which separates **how good the confidence ranking is** from **whether the validation threshold transfers**.

In [19]:
rows_ = []
for (regime, s), pl in PIPE.items():
    te = pl["test"]
    for cond, vs in CONDITIONS.items():
        m = te["variant"].isin(vs).to_numpy(); y = te.loc[m, "label"].to_numpy(); P = pl["P_test_cal"][m]
        pred, conf = pred_of(P), P.max(axis=1); acc = conf >= pl["tau"]
        ed = y == "EVENT_DRIFT"
        rows_.append({"regime": regime, "set": s, "condition": cond, "n": int(m.sum()), "tau": pl["tau"],
                      "coverage": acc.mean(), "selective_error": float((pred[acc] != y[acc]).mean()) if acc.any() else np.nan,
                      "target_risk": TARGET_RISK, "EVENT_DRIFT_accepted": float(acc[ed].mean()),
                      "macro_f1_all": f1_score(y, pred, average="macro", labels=LABELS, zero_division=0)})
GATE = pd.DataFrame(rows_)
print("Test results of the validation-selected gate (selective_error should be near or below target_risk if the threshold transfers):")
print(GATE.round(3).to_string(index=False))

rows_ = []
for (regime, s), pl in PIPE.items():
    te = pl["test"]
    for cond, vs in CONDITIONS.items():
        m = te["variant"].isin(vs).to_numpy(); y = te.loc[m, "label"].to_numpy(); P = pl["P_test_cal"][m]
        order = np.argsort(-P.max(axis=1), kind="stable"); wrong = (pred_of(P) != y)[order]
        rows_.append({"regime": regime, "set": s, "condition": cond, **{f"risk@{int(c * 100)}%": float(wrong[:max(1, int(round(c * len(wrong))))].mean()) for c in [0.2, 0.4, 0.6, 0.8, 1.0]}})
RISKCOV = pd.DataFrame(rows_)
print("\\nSelective error at fixed coverage on test (most confident windows first; lower is better):")
print(RISKCOV[RISKCOV["regime"] == "W"].drop(columns="regime").round(3).to_string(index=False))

Test results of the validation-selected gate (selective_error should be near or below target_risk if the threshold transfers):
regime set condition    n  tau  coverage  selective_error  target_risk  EVENT_DRIFT_accepted  macro_f1_all
     W   G  baseline 1108 0.65     0.269            0.282          0.2                 0.291         0.525
     W   G cool_ramp 1108 0.65     0.264            0.401          0.2                 0.257         0.524
     W   G warm_step 1108 0.65     0.282            0.228          0.2                 0.254         0.584
     W   G cool_step 1108 0.65     0.310            0.259          0.2                 0.362         0.559
     W   F  baseline 1108 0.62     0.453            0.201          0.2                 0.500         0.650
     W   F cool_ramp 1108 0.62     0.367            0.324          0.2                 0.317         0.381
     W   F warm_step 1108 0.62     0.478            0.087          0.2                 0.530         0.724
     W   F cool_s

## 8. Correction decision: correction rate and false-correction rate together
- **Correction rate** = share of true DRIFT windows that receive `correct`.
- **False-correction rate** (headline safety metric) = share of windows that are **not** DRIFT (NORMAL, EVENT, EVENT_DRIFT) that receive `correct`. It is shown with a 95% Wilson interval; windows overlap and come from only 4 test episodes, so the interval is optimistic.
- A gate that never corrects has a perfect false-correction rate and is useless, which is why both numbers sit side by side.

In [21]:
rows_ = []
for (regime, s), pl in PIPE.items():
    te = pl["test"]
    for cond, vs in CONDITIONS.items():
        m = te["variant"].isin(vs).to_numpy(); y = te.loc[m, "label"].to_numpy(); P = pl["P_test_cal"][m]
        dec = decide(P, pl["tau"], pl["t_corr"]); corr = dec == "correct"; non = y != "DRIFT"
        k, n = int(corr[non].sum()), int(non.sum()); lo, hi = wilson(k, n)
        rows_.append({"regime": regime, "set": s, "condition": cond, "t_corr": pl["t_corr"],
                      "correction_rate_on_DRIFT": float(corr[y == "DRIFT"].mean()),
                      "false_correction_rate": k / n, "FCR_95%_CI": f"{lo:.3f} to {hi:.3f}", "false_attempts": k, "non_drift_windows": n,
                      **{f"FCR_{c}": float(corr[y == c].mean()) for c in ["NORMAL", "EVENT", "EVENT_DRIFT"]},
                      "drift_ignored_rate": float(np.isin(dec[np.isin(y, ["DRIFT", "EVENT_DRIFT"])], ["no_action", "preserve"]).mean())})
CORR = pd.DataFrame(rows_)
print("Correction safety on test. 'drift_ignored_rate' = DRIFT or EVENT_DRIFT windows that were confidently given no_action or preserve:")
print(CORR.round(3).to_string(index=False))

Correction safety on test. 'drift_ignored_rate' = DRIFT or EVENT_DRIFT windows that were confidently given no_action or preserve:
regime set condition  t_corr  correction_rate_on_DRIFT  false_correction_rate     FCR_95%_CI  false_attempts  non_drift_windows  FCR_NORMAL  FCR_EVENT  FCR_EVENT_DRIFT  drift_ignored_rate
     W   G  baseline    0.52                     0.347                  0.052 0.039 to 0.070              44                840       0.007      0.011            0.146               0.043
     W   G cool_ramp    0.52                     0.347                  0.068 0.053 to 0.087              57                840       0.007      0.004            0.201               0.060
     W   G warm_step    0.52                     0.556                  0.118 0.098 to 0.141              99                840       0.007      0.011            0.351               0.000
     W   G cool_step    0.52                     0.556                  0.125 0.104 to 0.149             105          

In [22]:
# What happens to each true class? (regime W; share of that class's windows per decision)
DEC_ROWS = []
for (regime, s), pl in PIPE.items():
    te = pl["test"]
    for cond, vs in CONDITIONS.items():
        m = te["variant"].isin(vs).to_numpy(); y = te.loc[m, "label"].to_numpy()
        dec = decide(pl["P_test_cal"][m], pl["tau"], pl["t_corr"])
        ct = pd.crosstab(pd.Series(y, name="true"), pd.Series(dec, name="decision"), normalize="index").reindex(index=LABELS, columns=["no_action", "preserve", "correct", "defer_flag"], fill_value=0)
        for lab in LABELS:
            DEC_ROWS.append({"regime": regime, "set": s, "condition": cond, "true": lab, **ct.loc[lab].to_dict()})
DEC = pd.DataFrame(DEC_ROWS)
for s in SETS:
    for cond in ["baseline", "cool_ramp"]:
        print(f"\nRegime W, set {s}, condition {cond}: share of each true class per decision")
        print(DEC[(DEC["regime"] == "W") & (DEC["set"] == s) & (DEC["condition"] == cond)].drop(columns=["regime", "set", "condition"]).set_index("true").round(3).to_string())


Regime W, set G, condition baseline: share of each true class per decision
             no_action  preserve  correct  defer_flag
true                                                 
NORMAL           0.217     0.062    0.007       0.714
DRIFT            0.000     0.019    0.347       0.634
EVENT            0.037     0.235    0.011       0.716
EVENT_DRIFT      0.004     0.063    0.146       0.787

Regime W, set G, condition cool_ramp: share of each true class per decision
             no_action  preserve  correct  defer_flag
true                                                 
NORMAL           0.217     0.062    0.007       0.714
DRIFT            0.000     0.019    0.347       0.634
EVENT            0.082     0.205    0.004       0.709
EVENT_DRIFT      0.000     0.101    0.201       0.698

Regime W, set F, condition baseline: share of each true class per decision
             no_action  preserve  correct  defer_flag
true                                                 
NORMAL         

## 9. Save outputs (new folder; first NB6 outputs untouched)

In [24]:
CF.to_csv(OUT / "calibration_crossfit_validation.csv", index=False); CALT.to_csv(OUT / "calibration_test_quality.csv", index=False)
GATE.to_csv(OUT / "gate_test_results.csv", index=False); RISKCOV.to_csv(OUT / "risk_at_coverage_test.csv", index=False)
FRONT.to_csv(OUT / "validation_coverage_frontier.csv", index=False); CORR.to_csv(OUT / "correction_safety_test.csv", index=False); DEC.to_csv(OUT / "decision_shares_test.csv", index=False)
pred_rows = []
for (regime, s), pl in PIPE.items():
    for part, d, P in [("validation_crossfit", pl["val"], pl["P_val_cf"]), ("test", pl["test"], pl["P_test_cal"])]:
        df = d[["window_id", "episode_id", "variant", "label", "split"]].copy()
        df["regime"], df["set"], df["part"], df["calibrator"] = regime, s, part, pl["method"]
        for k, l in enumerate(LABELS):
            df[f"p_{l}"] = P[:, k]
        df["pred"], df["confidence"] = pred_of(P), P.max(axis=1)
        df["decision"] = decide(P, pl["tau"], pl["t_corr"])
        pred_rows.append(df)
pd.concat(pred_rows).to_parquet(OUT / "nb6_v2_predictions.parquet", index=False)
with open(OUT / "nb6_v2_config.json", "w") as f:
    json.dump({"sets": SETS, "regimes": ["W", "ALL"], "rf_params": {**RF_PARAMS, "n_estimators": N_TREES, "class_weight": "balanced", "seed": SEED},
               "calibration_methods": CAL_METHODS, "prob_floor": EPS, "target_risk": TARGET_RISK, "min_accepted": MIN_ACCEPTED,
               "max_false_correction_rate": MAX_FALSE_CORRECTION, "chosen": {f"{r}/{s}": {"calibrator": pl["method"], "tau": pl["tau"], "t_corr": pl["t_corr"]} for (r, s), pl in PIPE.items()},
               "conditions": CONDITIONS, "decision_policy": "correct: pred DRIFT and P(DRIFT)>=t_corr; no_action/preserve: confidence>=tau and pred NORMAL/EVENT; otherwise defer_flag",
               "quick_test": QUICK_TEST, "sklearn": sklearn.__version__}, f, indent=2, default=str)
print("saved to", OUT)

saved to C:\AIoT_Research\outputs\notebook06_v2


## 10. Summary and what must not be claimed

In [26]:
print("Chosen per regime/set (calibrator, tau, t_corr):")
for (r, s), pl in PIPE.items():
    print(f"  {r:>3}/{s}: {pl['method']:<11} tau={pl['tau']:.2f}  t_corr={pl['t_corr']:.2f}")
w = GATE[GATE["regime"] == "W"].pivot(index="condition", columns="set", values="coverage").reindex(list(CONDITIONS))
print("\nTest coverage of the abstention gate, regime W:"); print(w.round(3).to_string())
w = CORR[CORR["regime"] == "W"].pivot(index="condition", columns="set", values="false_correction_rate").reindex(list(CONDITIONS))
print("\nTest false-correction rate, regime W:"); print(w.round(3).to_string())
w = CORR[CORR["regime"] == "W"].pivot(index="condition", columns="set", values="correction_rate_on_DRIFT").reindex(list(CONDITIONS))
print("\nTest correction rate on DRIFT windows, regime W (a false-correction rate is only meaningful next to this):"); print(w.round(3).to_string())
print("\nDo NOT claim:")
print(" - a low false-correction rate, unless the correction rate beside it is meaningfully above zero;")
print(" - that calibration helped, unless section 6 shows it beating raw and uniform;")
print(" - that thresholds transfer to cooling or step data, unless the gate table shows selective error near the target there too;")
print(" - anything about feature set I as a result: it was added after seeing NB5 v2 and is exploratory.")
print("Limits: 4 test episodes; overlapping windows; synthetic injections; voltage shift unhandled; no correction magnitude is evaluated here (NB7).")

Chosen per regime/set (calibrator, tau, t_corr):
    W/G: raw         tau=0.65  t_corr=0.52
    W/F: raw         tau=0.62  t_corr=0.49
    W/I: raw         tau=0.67  t_corr=0.40
  ALL/G: platt       tau=0.70  t_corr=0.30
  ALL/F: platt       tau=0.72  t_corr=0.45
  ALL/I: platt       tau=0.72  t_corr=0.47

Test coverage of the abstention gate, regime W:
set            F      G      I
condition                     
baseline   0.453  0.269  0.291
cool_ramp  0.367  0.264  0.288
warm_step  0.478  0.282  0.292
cool_step  0.413  0.310  0.320

Test false-correction rate, regime W:
set            F      G      I
condition                     
baseline   0.046  0.052  0.058
cool_ramp  0.240  0.068  0.043
warm_step  0.096  0.118  0.118
cool_step  0.255  0.125  0.064

Test correction rate on DRIFT windows, regime W (a false-correction rate is only meaningful next to this):
set            F      G      I
condition                     
baseline   0.646  0.347  0.593
cool_ramp  0.646  0.347  0.593
w